<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/Qwen3_4B_WANDA_pruned_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# ⚙️ Installation & Setup
# - Transformers >= 4.51.0, weil Qwen3 sonst nicht geladen wird (siehe Model Card)
# - Keine Trainings-Loops, nur Forward-Pässe für WANDA-Calibration
!pip -q install "transformers>=4.51.0" "accelerate>=0.33.0" "torch>=2.1" sentencepiece

import os, json, math, random, gc
from dataclasses import dataclass
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# %% [markdown]
# 🔗 Google Drive mounten & Pfade setzen
from google.colab import drive
drive.mount('/content/drive')

TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/12B_trainingdata.json"
TEST_PATH  = "/content/drive/MyDrive/Colab Notebooks/12B_golden_testdata.json"

# Falls Trainingsdatei (zu groß) nicht vorhanden ist, nutzen wir Test für Calibration als Fallback.
if not os.path.exists(TRAIN_PATH):
    if os.path.exists(TEST_PATH):
        TRAIN_PATH = TEST_PATH
        print("⚠️ Trainingsdatei nicht gefunden – nutze Testdaten als Calibration-Fallback.")
    else:
        raise FileNotFoundError("Weder Trainings- noch Testdatei gefunden.")

# %% [markdown]
# 🧾 Datensatz laden & schlanke Calibration-Beispiele bauen
def load_bioasq_questions(path: str) -> List[Dict]:
    with open(path, "r") as f:
        data = json.load(f)
    # Struktur: {"questions":[{ "body": "...", "type": "...", ...}, ...]}
    return data["questions"]

all_questions = load_bioasq_questions(TRAIN_PATH)
random.seed(42)
random.shuffle(all_questions)

# Kleine Calibration-Menge wählen (speicherschonend)
CALIB_SAMPLES = 64      # bei OOM auf 32/16 reduzieren
MAX_PROMPT_TOKENS = 256 # kurzer Kontext hält Speicherbedarf gering

def build_prompt(q):
    # Ein kompakter, einheitlicher Prompt – wir brauchen nur natürliche Aktivierungen,
    # keine "thinking"-Sequenzen o.ä. für Calibration.
    t = q.get("type", "")
    body = q.get("body", "").strip()
    head = "Beantworte medizinische Frage knapp und sachlich."
    if t == "yesno":
        instr = "Antworte mit Ja oder Nein und maximal einem kurzen Satz."
    elif t == "list":
        instr = "Gib eine knappe Liste der wichtigsten Punkte aus."
    elif t == "factoid":
        instr = "Gib die gesuchte Entität in 1–2 Wörtern an."
    else:
        instr = "Antworte kurz."
    return f"{head}\nFrage: {body}\n{instr}"

calib_texts = [build_prompt(q) for q in all_questions[:CALIB_SAMPLES]]

# %% [markdown]
# ⬇️ Qwen3-4B laden (Instruct). Bei Bedarf könnt ihr auch Qwen/Qwen3-4B-Base nehmen.
MODEL_ID = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# "thinking" wollen wir für Calibration vermeiden -> enable_thinking=False in Chat-Template.
def encode_batch(texts: List[str], max_len=MAX_PROMPT_TOKENS):
    # Qwen3 nutzt Chat-Templates; wir simulieren USER-Nachricht ohne Thinking
    rendered = [
        tokenizer.apply_chat_template(
            [{"role":"user","content":t}],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        for t in texts
    ]
    toks = tokenizer(
        rendered, return_tensors="pt",
        padding=True, truncation=True, max_length=max_len
    )
    return toks

# Speicherfreundliches Laden (FP16 + device_map="auto" für ggf. CPU-Offloading)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False  # spart RAM beim Forward

# %% [markdown]
# 🪄 WANDA-Calibration: Eingangsaktivierungs-Normen sammeln (pro nn.Linear)
@dataclass
class LayerStats:
    sumsq: torch.Tensor  # Summe der Quadrate pro Eingangsfeature (auf CPU)
    in_features: int

def iter_linear_modules(model: nn.Module):
    # Wir prunen alle nn.Linear außer Embedding/Langzeitkopf
    skip_names = ("lm_head", "embed_tokens")
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and not any(s in name for s in skip_names):
            yield name, module

# Stats-Speicher vorbereiten
layer_stats: Dict[str, LayerStats] = {}
for name, mod in iter_linear_modules(model):
    layer_stats[name] = LayerStats(
        sumsq=torch.zeros(mod.in_features, dtype=torch.float64, device="cpu"),
        in_features=mod.in_features
    )

# Hook-Funktion: akkumuliere Summe der Quadrate über Batch*Seq
def make_hook(name: str):
    def hook(module, inputs, output):
        # inputs[0] Form: (B, L, C_in) oder (B*L, C_in)
        x = inputs[0]
        if x.dim() == 3:
            x = x.reshape(-1, x.size(-1))
        # auf GPU berechnen, dann auf CPU addieren (nur Vektorgröße C_in)
        ss = (x.float() ** 2).sum(dim=0).detach().cpu()
        layer_stats[name].sumsq += ss
        return None
    return hook

# Hooks registrieren
handles = []
for name, mod in iter_linear_modules(model):
    handles.append(mod.register_forward_hook(make_hook(name)))

# Calibration laufen lassen – kleine Batches, kein KV-Cache, no_grad
BATCH_SIZE = 2  # bei OOM auf 1 reduzieren
with torch.no_grad():
    for i in range(0, len(calib_texts), BATCH_SIZE):
        batch = calib_texts[i:i+BATCH_SIZE]
        toks = encode_batch(batch)
        toks = {k: v.to(model.device) for k, v in toks.items()}
        _ = model(**toks)  # nur Forward, keine Generation
        del toks
        torch.cuda.empty_cache()

# Hooks entfernen
for h in handles:
    h.remove()
del handles
gc.collect()
torch.cuda.empty_cache()

# %% [markdown]
# ✂️ WANDA-Pruning anwenden (unstrukturiert, per Output/Zeile)
SPARSITY = 0.50  # 50% der Gewichte pro Zeile auf 0. Nach Bedarf anpassen (0.3..0.6 als Startpunkte).
EPS = 1e-12

total_params = 0
total_zero = 0

for name, mod in iter_linear_modules(model):
    W = mod.weight.data  # (C_out, C_in)
    Cout, Cin = W.shape
    stats = layer_stats[name]
    # L2-Norm über Tokens (√(Summe Quadrate))
    act_norm = torch.sqrt(stats.sumsq + EPS)  # (C_in,)
    act_norm = act_norm.to(W.device, dtype=torch.float32)

    # Wanda-Metrik: |W| * ||X||_2, pro Output vergleichen
    metric = W.abs().to(torch.float32) * act_norm.unsqueeze(0)  # (C_out, C_in)

    k_prune = int(Cin * SPARSITY)
    if k_prune == 0:
        continue

    # kleinste Scores je Zeile finden
    prune_idx = torch.argsort(metric, dim=1)[:, :k_prune]  # (C_out, k)
    # Maske bauen: True=behalten, False=prunen
    mask = torch.ones_like(W, dtype=torch.bool, device=W.device)
    mask.scatter_(1, prune_idx, False)
    # Anwenden
    W[~mask] = 0

    total_params += W.numel()
    total_zero   += (~mask).sum().item()

    # Optional: Bias nicht anfassen
    # (Qwen-Linear-Schichten sind typischerweise bias=False)

sparsity_achieved = total_zero / max(1, total_params)
print(f"✅ Pruning fertig. Globale (unstrukturierte) Sparsity: {sparsity_achieved:.2%}")

# %% [markdown]
# 💾 Gepruntes Modell speichern
OUT_DIR = f"/content/drive/MyDrive/Colab Notebooks/qwen3-4b-wanda-{int(SPARSITY*100)}p"
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR)
print("Gespeichert nach:", OUT_DIR)

# %% [markdown]
# 🔎 Mini-Sanity-Check (kurze Generierung auf 1 Beispiel, niedrige Tokens, um OOM zu vermeiden)
example = build_prompt(all_questions[CALIB_SAMPLES]) if len(all_questions) > CALIB_SAMPLES else "Nenne die Symptome einer leichten Erkältung in 3 Stichpunkten."
inputs = tokenizer.apply_chat_template([{"role":"user","content":example}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
toks = tokenizer([inputs], return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**toks, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
# %% [markdown]
# 📦 Installation (falls noch nicht geschehen) & Imports
!pip -q install "transformers>=4.51.0" "accelerate>=0.33.0" "torch>=2.1" pandas sentencepiece

import os, json, math, re, random, gc
from collections import Counter
from typing import List, Dict, Tuple, Set

import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# %% [markdown]
# 🔗 Drive mounten & Pfade
from google.colab import drive
drive.mount('/content/drive')

TEST_PATH  = "/content/drive/MyDrive/Colab Notebooks/12B_golden_testdata.json"
# ⬇️ Pfad zu DEINEM geprunten Modell anpassen, falls nötig
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/qwen3-4b-wanda-50p"

assert os.path.exists(TEST_PATH), f"Testdatei nicht gefunden: {TEST_PATH}"
assert os.path.exists(MODEL_PATH), f"Gepruntes Modell nicht gefunden: {MODEL_PATH}"

# %% [markdown]
# 📥 Daten laden
with open(TEST_PATH, "r") as f:
    ds = json.load(f)
questions = ds["questions"]
counts = Counter(q["type"] for q in questions)
print("Fragetypen:", dict(counts))

# %% [markdown]
# 🧹 Normalisierung & Matching-Helfer (BioASQ-Style, aber leichtgewichtig)
def normalize(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    # Satzzeichen raus (einfacher Normalizer)
    s = re.sub(r"[^\w\s\-/%]", "", s)
    s = s.strip(" .,:;!?\"'()[]{}")
    return s

def norm_set(items: List[str]) -> Set[str]:
    return {normalize(x) for x in items if str(x).strip()}

def match_any(pred: str, gold_syns: List[str]) -> bool:
    p = normalize(pred)
    gset = norm_set(gold_syns)
    return p in gset

def split_pred_list(text: str) -> List[str]:
    # Liste aus Modellantwort zerlegen: Komma/Strichpunkt/Newline
    parts = re.split(r"[,\n;]+", text)
    parts = [p.strip() for p in parts if p.strip()]
    # Doppelte & sehr kurze Artefakte verwerfen
    uniq = []
    seen = set()
    for p in parts:
        n = normalize(p)
        if len(n) == 0 or n in seen:
            continue
        seen.add(n)
        uniq.append(p)
    return uniq

# %% [markdown]
# 🧠 Modell laden (FP16, Auto-Offload), Qwen3 Chat-Template
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

def render_user(msg: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role":"user","content":msg}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

def generate_once(prompt: str, max_new_tokens=8, do_sample=False, temperature=0.7, top_p=0.9) -> str:
    text = render_user(prompt)
    toks = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **toks,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )
    ans = tokenizer.decode(out[0], skip_special_tokens=True)
    return ans.split(text)[-1].strip()  # nur die Assistant-Antwort

# %% [markdown]
# 📐 Evaluations-Funktionen (Yes/No, Factoid, List)

# --- YES/NO ---
def yn_label_from_text(s: str) -> str:
    s = normalize(s)
    # robustes Mapping
    if s.startswith("yes") or s in {"y", "yeah", "yep", "true"}:
        return "yes"
    if s.startswith("no") or s in {"n", "nope", "false"}:
        return "no"
    # fallback: heuristik (positives Wort)
    return "yes" if "yes" in s else "no" if "no" in s else "no"

def f1(p, r):
    return 0.0 if (p+r)==0 else 2*p*r/(p+r)

def eval_yesno(preds: List[str], golds: List[str]) -> Dict[str, float]:
    # Klassenweise Counts
    labels = ["yes","no"]
    cm = {c: {"tp":0, "fp":0, "fn":0, "tn":0} for c in labels}
    correct = 0
    for pr, gd in zip(preds, golds):
        if pr == gd: correct += 1
        for c in labels:
            tp = (pr==c and gd==c)
            fp = (pr==c and gd!=c)
            fn = (pr!=c and gd==c)
            tn = (pr!=c and gd!=c)
            cm[c]["tp"] += tp
            cm[c]["fp"] += fp
            cm[c]["fn"] += fn
            cm[c]["tn"] += tn
    acc = correct/len(golds) if golds else 0.0
    out = {"Accuracy": acc}
    f1s = []
    for c in labels:
        P = cm[c]["tp"] / max(1, (cm[c]["tp"]+cm[c]["fp"]))
        R = cm[c]["tp"] / max(1, (cm[c]["tp"]+cm[c]["fn"]))
        out[f"F1 {c.capitalize()}"] = f1(P,R)
        f1s.append(out[f"F1 {c.capitalize()}"])
    out["Macro F1"] = sum(f1s)/len(f1s)
    return out

# --- FACTOID ---
def eval_factoid(pred_topk: List[List[str]], gold_groups: List[List[str]]) -> Dict[str,float]:
    # gold_groups: z.B. [["myocardial infarction","heart attack"]] = Synonyme einer Entität
    # pred_topk: Liste von Kandidaten-Strings nach Rang
    strict_hits = 0
    lenient_hits = 0
    rr_sum = 0.0
    n = len(pred_topk)
    for cand_list, gold in zip(pred_topk, gold_groups):
        # flatten gold-synonyme (alle Gruppen zusammen, es gibt 1 gold-item mit syns)
        gold_syns = [g for g in gold] if isinstance(gold[0], str) else [x for group in gold for x in group]
        rank = None
        for i, cand in enumerate(cand_list, start=1):
            if match_any(cand, gold_syns):
                rank = i
                break
        if rank == 1: strict_hits += 1
        if rank is not None:
            lenient_hits += 1
            rr_sum += 1.0/rank
    return {
        "Strict Acc.": strict_hits/max(1,n),
        "Lenient Acc.": lenient_hits/max(1,n),
        "MRR": rr_sum/max(1,n),
    }

# --- LIST ---
def eval_list(pred_lists: List[List[str]], gold_groups_list: List[List[List[str]]]) -> Dict[str,float]:
    # gold_groups_list: pro Frage Liste von Gruppen (Synonyme)
    precs, recs, fms = [], [], []
    for preds, gold_groups in zip(pred_lists, gold_groups_list):
        gold_matched = [False]*len(gold_groups)
        # zähle Treffer, ohne doppelt dieselbe Gruppe zu matchen
        correct = 0
        for p in preds:
            hit = False
            for gi, group in enumerate(gold_groups):
                if gold_matched[gi]:
                    continue
                if match_any(p, group):
                    gold_matched[gi] = True
                    hit = True
                    break
            if hit:
                correct += 1
        P = correct / max(1, len(preds))
        R = correct / max(1, len(gold_groups))
        precs.append(P); recs.append(R); fms.append(f1(P,R))
    return {
        "Mean Prec.": sum(precs)/max(1,len(precs)),
        "Recall":     sum(recs)/max(1,len(recs)),
        "F-Measure":  sum(fms)/max(1,len(fms)),
    }

# %% [markdown]
# 🧪 Vorhersagen erzeugen (sparsam!)
random.seed(42)
YESNO_MAX_NEW = 2
FACTOID_MAX_NEW = 8
LIST_MAX_NEW = 24
FACTOID_TOPK = 5  # für Lenient & MRR

yn_gold, yn_pred = [], []
fact_gold, fact_pred_topk = [], []
list_gold, list_pred = [], []

for q in questions:
    qtype = q["type"]
    body  = q["body"].strip()

    if qtype == "yesno":
        gold = q["exact_answer"].strip().lower()  # 'yes' | 'no'
        prompt = (
            "Answer the biomedical yes/no question with a single word ONLY: 'yes' or 'no'.\n"
            f"Question: {body}\n"
            "Answer:"
        )
        out = generate_once(prompt, max_new_tokens=YESNO_MAX_NEW, do_sample=False)
        pr = yn_label_from_text(out)
        yn_gold.append(gold); yn_pred.append(pr)

    elif qtype == "factoid":
        gold_groups = q["exact_answer"]  # [["syn1","syn2",...]]
        prompt = (
            "Answer the biomedical factoid question with ONLY the short entity name (1-3 words). "
            "No extra text, no punctuation.\n"
            f"Question: {body}\n"
            "Short answer:"
        )
        # Top-1 (greedy)
        topk = []
        out1 = generate_once(prompt, max_new_tokens=FACTOID_MAX_NEW, do_sample=False)
        topk.append(out1)
        # Weitere Kandidaten mit Sampling (temperatur), Duplikate vermeiden
        for k in range(FACTOID_TOPK-1):
            outk = generate_once(prompt, max_new_tokens=FACTOID_MAX_NEW, do_sample=True, temperature=0.8, top_p=0.9)
            if normalize(outk) not in {normalize(x) for x in topk}:
                topk.append(outk)
        fact_gold.append(gold_groups)
        fact_pred_topk.append(topk[:FACTOID_TOPK])

    elif qtype == "list":
        gold_groups = q["exact_answer"]  # [[syns of item1], [syns of item2], ...]
        prompt = (
            "List the biomedical items asked for, as a comma-separated list. "
            "Return ONLY the list, no explanations.\n"
            f"Question: {body}\n"
            "List:"
        )
        out = generate_once(prompt, max_new_tokens=LIST_MAX_NEW, do_sample=False)
        preds = split_pred_list(out)
        list_gold.append(gold_groups)
        list_pred.append(preds)

    else:
        # 'summary' o.ä. – für diese Tabelle nicht evaluiert
        continue

    # Speicher freigeben
    torch.cuda.empty_cache()

print(f"Yes/No eval: {len(yn_gold)} Fragen | Factoid eval: {len(fact_gold)} | List eval: {len(list_gold)}")

# %% [markdown]
# 🧮 Metriken berechnen
metrics = {}
if yn_gold:
    metrics.update(eval_yesno(yn_pred, yn_gold))
if fact_gold:
    metrics.update(eval_factoid(fact_pred_topk, fact_gold))
if list_gold:
    metrics.update(eval_list(list_pred, list_gold))

# %% [markdown]
# 🗃️ Tabelle mit Baseline + Geprunt
# Baseline-Werte deines Kommilitonen (DE-Komma -> float)
baseline = {
    "System": "BioASQ_Baseline",
    "Accuracy": 0.4706,
    "F1 Yes": 0.4000,
    "F1 No": 0.5263,
    "Macro F1": 0.4632,
    "Strict Acc.": 0.1538,
    "Lenient Acc.": 0.2692,
    "MRR": 0.1955,
    "Mean Prec.": 0.2503,
    "Recall": 0.2390,
    "F-Measure": 0.2202,
}

pruned_row = {
    "System": "Qwen3-4B-WANDA",
    "Accuracy": round(metrics.get("Accuracy", 0.0), 4),
    "F1 Yes": round(metrics.get("F1 Yes", 0.0), 4),
    "F1 No": round(metrics.get("F1 No", 0.0), 4),
    "Macro F1": round(metrics.get("Macro F1", 0.0), 4),
    "Strict Acc.": round(metrics.get("Strict Acc.", 0.0), 4),
    "Lenient Acc.": round(metrics.get("Lenient Acc.", 0.0), 4),
    "MRR": round(metrics.get("MRR", 0.0), 4),
    "Mean Prec.": round(metrics.get("Mean Prec.", 0.0), 4),
    "Recall": round(metrics.get("Recall", 0.0), 4),
    "F-Measure": round(metrics.get("F-Measure", 0.0), 4),
}

cols = ["System","Accuracy","F1 Yes","F1 No","Macro F1","Strict Acc.","Lenient Acc.","MRR","Mean Prec.","Recall","F-Measure"]
df = pd.DataFrame([baseline, pruned_row], columns=cols)

# 🇩🇪 Anzeige mit Komma als Dezimaltrenner
def format_de(x):
    if isinstance(x, float):
        return f"{x:.4f}".replace(".", ",")
    return x

df_display = df.copy()
for c in cols[1:]:
    df_display[c] = df_display[c].map(format_de)

print("\n✅ BioASQ-Metriken (Baseline vs. Geprunt):")
df_display
